## Data Mapping for GeoInfo

In [100]:
import sys, os
sys.path.insert(0, "/Users/trivenidhamdhere/Documents/MSc Data Science/MSc Dissertation - AZ/src/scripts")

import re
import pandas as pd
from data_utils import load_clean_parquets, preview

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

In [101]:
tables = load_clean_parquets()
hpa_rna = tables["hpa_rna"]
depmap_expr = tables["depmap_expr"]
geo_expr = tables["geo_expr"]
proteomics = tables["proteomics"]
protein_map = tables["protein_map"]
fusions = tables["fusions"]
mutations = tables["mutations"]
cellosaurus = tables["cellosaurus"]
depmap_profiles = tables["depmap_profiles"]
sample_info = tables["sample_info"]
geo_info = tables["geo_info"]
hpa_desc = tables["hpa_desc"]
metabolomics = tables["metabolomics"]
mirna = tables["mirna"]
signatures = tables["signatures"]

Loaded hpa_rna: (24315372, 6)
Loaded depmap_expr: (1495, 53961)
Loaded geo_expr: (19914, 3268)
Loaded proteomics: (375, 12559)
Loaded protein_map: (12558, 3)
Loaded fusions: (184237, 32)
Loaded mutations: (1066869, 70)
Loaded cellosaurus: (152231, 17)
Loaded depmap_profiles: (3830, 5)
Loaded sample_info: (1840, 29)
Loaded geo_info: (3267, 23)
Loaded hpa_desc: (1206, 7)
Loaded metabolomics: (928, 227)
Loaded mirna: (734, 956)
Loaded signatures: (3021, 12)


In [102]:
import numpy as np
import pandas as pd

def audit(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    miss = df.isna().sum()
    pct = (miss.to_numpy(dtype=float) / n * 100).round(1) if n else np.zeros(len(miss))
    return (pd.DataFrame({
        "column":    miss.index,
        "dtype":     df.dtypes.astype(str).to_numpy(),
        "missing_n": miss.to_numpy(),
        "missing_%": pct,
    })
    .sort_values("missing_%", ascending=False)
    .reset_index(drop=True))

# Geo_Info

In [103]:
geo_info

,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_institute,gse_id,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type,cell_line_trimmed
0,gsm101610,gsm101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0131,a-172,cello geo gsm,NaN
1,gsm101615,gsm101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln-229,cello geo gsm,NaN
2,gsm101616,gsm101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln-229,cello geo gsm,NaN
3,gsm101667,gsm101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,NaN
4,gsm101668,gsm101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262,gsm960294,gsm960294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN
3263,gsm960295,gsm960295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN
3264,gsm960296,gsm960296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN
3265,gsm960297,gsm960297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_7082,NaN,NaN,NaN


In [104]:
import re

# canonical normaliser: lowercase, then strip ALL whitespace + separators
# (removing spaces too, so "MDA-MB 231" and "mda_mb_231" collapse to the same key)
_sep = re.compile(r"[\s,\-_/]+")

def norm_cellline(s: pd.Series) -> pd.Series:
    return (s.astype("string")
             .str.lower()
             .str.replace(_sep, "", regex=True)   # kills spaces, , - _ /
             .str.strip())                          # tidy any residue

geo_info["cellline"] = norm_cellline(geo_info["cellline"])
geo_info = geo_info.drop(columns=["cell_line_trimmed"], errors="ignore")

In [105]:
audit(geo_info)

,column,dtype,missing_n,missing_%
0,disease,str,2443,74.8
1,origin,str,2340,71.6
2,cell_line,str,938,28.7
3,gse_filename,str,938,28.7
4,gse_id,str,938,28.7
5,contact_institute,str,938,28.7
6,contact_country,str,938,28.7
7,characteristics_ch1,str,938,28.7
8,platform_id,str,938,28.7
9,source_name_ch1,str,938,28.7


In [106]:
sample_info

,depmap_id,cell_line_name,stripped_cell_line_name,ccle_name,alias,cosmicid,sex,source,rrid,wtsi_master_cell_id,...,lineage_sub_subtype,lineage_molecular_subtype,default_growth_pattern,model_manipulation,model_manipulation_details,patient_id,parent_depmap_id,cellosaurus_ncit_disease,cellosaurus_ncit_id,cellosaurus_issues
0,ach-000016,slr 21,slr21,slr21_kidney,NaN,NaN,NaN,academic lab,cvcl_v607,NaN,...,NaN,NaN,NaN,NaN,NaN,pt-jnarlb,NaN,clear cell renal cell carcinoma,c4033,NaN
1,ach-000032,mhh-call-3,mhhcall3,mhhcall3_haematopoietic_and_lymphoid_tissue,NaN,NaN,female,dsmz,cvcl_0089,NaN,...,b_cell,NaN,NaN,NaN,NaN,pt-p2koyi,NaN,childhood b acute lymphoblastic leukemia,c9140,NaN
2,ach-000033,nci-h1819,ncih1819,ncih1819_lung,NaN,NaN,female,academic lab,cvcl_1497,NaN,...,nsclc_adenocarcinoma,NaN,NaN,NaN,NaN,pt-9p1wqv,NaN,lung adenocarcinoma,c3512,NaN
3,ach-000043,hs 895.t,hs895t,hs895t_fibroblast,NaN,NaN,female,atcc,cvcl_0993,NaN,...,NaN,NaN,2d: adherent,NaN,NaN,pt-rtuvzq,NaN,melanoma,c3224,NaN
4,ach-000049,hek te,hekte,hekte_kidney,NaN,NaN,NaN,academic lab,cvcl_ws59,NaN,...,NaN,NaN,NaN,immortalized,NaN,pt-qwyygr,NaN,NaN,NaN,no information is available about this cell li...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1835,ach-002393,cro-ap3,croap3,croap3_haematopoietic_and_lymphoid_tissue,NaN,NaN,male,sanger,cvcl_1810,NaN,...,b_cell_primary_effusion,NaN,NaN,NaN,NaN,pt-tc0lzm,NaN,primary effusion lymphoma,c6915,NaN
1836,ach-002394,geo,geo,geo_large_intestine,NaN,NaN,NaN,sanger,cvcl_0271,NaN,...,NaN,NaN,NaN,NaN,NaN,pt-fa1q9q,NaN,colon carcinoma,c4910,NaN
1837,ach-002395,huh-6 clone 5,huh6clone5,huh6clone5_liver,NaN,NaN,male,sanger,cvcl_1296,NaN,...,NaN,NaN,NaN,NaN,NaN,pt-ttixsl,ach-000671,hepatoblastoma,c3728,NaN
1838,ach-002396,sarc9371,sarc9371,sarc9371_bone,NaN,NaN,NaN,sanger,cvcl_5g89,NaN,...,NaN,NaN,NaN,NaN,NaN,pt-715fdc,NaN,osteosarcoma,c9145,NaN


In [107]:
# --- normalise both key sets to bare lowercase cvcl_xxxx ---
rrid_col = next(c for c in sample_info.columns if "rrid" in c.lower())

geo_cvcl = set(geo_info["cellosaurus_id"].dropna().str.strip().str.lower())
si_cvcl  = set(sample_info[rrid_col].dropna()
               .str.replace(r"(?i)^rrid:", "", regex=True).str.strip().str.lower())

overlap   = geo_cvcl & si_cvcl
only_geo  = geo_cvcl - si_cvcl

print(f"geo cvcl (unique):        {len(geo_cvcl)}")
print(f"sample_info cvcl (unique):{len(si_cvcl)}")
print(f"overlap:                  {len(overlap)}  ({len(overlap)/len(geo_cvcl)*100:.1f}%)")
print(f"in geo but NOT sample_info:{len(only_geo)}")

geo cvcl (unique):        797
sample_info cvcl (unique):1814
overlap:                  588  (73.8%)
in geo but NOT sample_info:209


In [108]:
geo_info

,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,contact_country,contact_institute,gse_id,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type
0,gsm101610,gsm101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0131,a172,cello geo gsm
1,gsm101615,gsm101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm
2,gsm101616,gsm101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm
3,gsm101667,gsm101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm
4,gsm101668,gsm101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262,gsm960294,gsm960294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm
3263,gsm960295,gsm960295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm
3264,gsm960296,gsm960296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm
3265,gsm960297,gsm960297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,cvcl_7082,<NA>,NaN


In [109]:
rrid_col = next(c for c in sample_info.columns if "rrid" in c.lower())
id_col   = next(c for c in sample_info.columns if c in ("depmap_id","modelid","model_id"))

# normalised cvcl -> model_id lookup from sample_info
si = sample_info[[rrid_col, id_col]].copy()
si["cvcl_key"] = (si[rrid_col].str.replace(r"(?i)^rrid:", "", regex=True)
                  .str.strip().str.lower())
si["model_id_new"] = si[id_col].str.strip().str.lower()
si = si.dropna(subset=["cvcl_key"]).drop_duplicates("cvcl_key")

# key on geo side
geo_info["cvcl_key"] = geo_info["cellosaurus_id"].str.strip().str.lower()

# map model_id in
geo_info = geo_info.merge(
    si[["cvcl_key", "model_id_new"]], on="cvcl_key", how="left"
)

# if geo_info already had a model_id column, reconcile; else just rename
if "model_id" in geo_info.columns:
    # fill any gaps, and check the new mapping agrees with the old where both exist
    both = geo_info["model_id"].notna() & geo_info["model_id_new"].notna()
    disagree = (geo_info.loc[both, "model_id"] != geo_info.loc[both, "model_id_new"]).sum()
    print(f"rows with old & new model_id: {both.sum()} | disagreements: {disagree}")
    geo_info["model_id"] = geo_info["model_id"].fillna(geo_info["model_id_new"])
else:
    geo_info = geo_info.rename(columns={"model_id_new": "model_id"})

geo_info = geo_info.drop(columns=[c for c in ["cvcl_key","model_id_new"]
                                  if c in geo_info.columns])

# --- report ---
resolved = geo_info["model_id"].notna().sum()
print(f"geo rows: {len(geo_info)} | with model_id: {resolved} "
      f"({resolved/len(geo_info)*100:.1f}%)")
print(f"distinct model_ids: {geo_info['model_id'].nunique()}")

geo rows: 3267 | with model_id: 2308 (70.6%)
distinct model_ids: 588


In [110]:
# model_id already filled by the earlier merge; just flag join status for provenance
geo_info["omics_linked"] = geo_info["model_id"].notna()

print(geo_info["omics_linked"].value_counts())
print(f"\nlinked to DepMap omics: {geo_info['omics_linked'].sum()} rows")
print(f"expression-only (no DepMap profile): {(~geo_info['omics_linked']).sum()} rows")

omics_linked
True     2308
False     959
Name: count, dtype: int64

linked to DepMap omics: 2308 rows
expression-only (no DepMap profile): 959 rows


In [111]:
geo_info


,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,gse_id,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type,model_id,omics_linked
0,gsm101610,gsm101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_0131,a172,cello geo gsm,ach-000558,True
1,gsm101615,gsm101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm,ach-000595,True
2,gsm101616,gsm101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm,ach-000595,True
3,gsm101667,gsm101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,ach-000437,True
4,gsm101668,gsm101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,ach-000437,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262,gsm960294,gsm960294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN,False
3263,gsm960295,gsm960295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN,False
3264,gsm960296,gsm960296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN,False
3265,gsm960297,gsm960297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,cvcl_7082,<NA>,NaN,NaN,False


In [112]:
na_model = geo_info[geo_info["model_id"].isna()]

print(f"rows with NA model_id: {len(na_model)} / {len(geo_info)}")
print(f"unique cvcl among them: {na_model['cellosaurus_id'].nunique()}")
print(f"of those, rows that HAD a cvcl but still no model: "
      f"{na_model['cellosaurus_id'].notna().sum()}")   # unmatched in sample_info
print(f"rows with no cvcl at all: {na_model['cellosaurus_id'].isna().sum()}")

# preview
na_model[["cellline", "cellosaurus_id", "cell_line"]].drop_duplicates().head(30)

rows with NA model_id: 959 / 3267
unique cvcl among them: 209
of those, rows that HAD a cvcl but still no model: 851
rows with no cvcl at all: 108


,cellline,cellosaurus_id,cell_line
31,hacat,cvcl_0038,hacat
40,hs68,cvcl_0839,hs68
46,kpl4,cvcl_5310,kpl-4
83,<NA>,cvcl_7082,NaN
117,u2932,cvcl_1896,NaN
182,uiso,cvcl_e996,uiso
188,waga,cvcl_e998,waga
236,<NA>,NaN,zrt
432,hec88nu,cvcl_2932,hec-88nu
459,ecc1,cvcl_7260,ecc-1


In [113]:
# rows with no cvcl but a raw cell_line name -> try to match into cellosaurus
no_cvcl = geo_info[geo_info["model_id"].isna() & geo_info["cellosaurus_id"].isna()]
recoverable = no_cvcl[no_cvcl["cell_line"].notna()]
print(f"no-cvcl rows: {len(no_cvcl)} | with a raw name to try: {len(recoverable)}")
print("raw names to attempt:", sorted(recoverable["cell_line"].dropna().unique()))

# build a normalised name index over cellosaurus (primary name + synonyms if present)
name_cols = [c for c in cellosaurus.columns
             if "name" in c or "synonym" in c or c in ("id","sy")]
cvcl_col  = next(c for c in cellosaurus.columns if "cvcl" in c or "accession" in c)
print("\ncellosaurus name columns available:", name_cols, "| cvcl col:", cvcl_col)

no-cvcl rows: 108 | with a raw name to try: 108
raw names to attempt: ['a4_fukada', 'bj', 'bt474ei', 'bt5491', 'c106', 'cal1201', 'cc20', 'd22', 'd24', 'd38', 'evsa1', 'faragex1', 'gam016', 'gam022', 'gs-1-2', 'gs-10', 'gs-11', 'gs-12', 'gs-2-2', 'gs-3', 'gs-3-2', 'gs-4', 'gs-4-2', 'gs-5', 'gs-5-2', 'gs-6', 'gs-7', 'gs-7-2', 'gs-8', 'gs-8-2', 'gs-9', 'gs-9-2', 'hotz', 'l115clone1', 'lb61', 'lncapcasres', 'lncapcasresnewcastle', 'mcf7mdrpos', 'mcg803', 'ml-4', 'ml-9', 'ncih366', 'ncih460dnp53', 'sd148', 'skbr31', 't24neo1', 't24neo3', 't24r2wtk10', 't24sauvage', 't24wtk6', 't47d1', 'zl25', 'zr75', 'zrt']

cellosaurus name columns available: ['cellosaurus_cell_line_name', 'synonyms'] | cvcl col: cellosaurus_accession


In [114]:
import re
_bracket = re.compile(r"[\[\(\{].*?[\]\)\}]")
_sep     = re.compile(r"[\s,\-_/.;:]+")
def _norm(s):
    return (s.astype("string").str.lower()
             .str.replace(_bracket,"",regex=True)
             .str.replace(_sep,"",regex=True).str.strip())

# explode cellosaurus synonyms into a long name->cvcl lookup
cello = cellosaurus[["cellosaurus_accession",
                     "cellosaurus_cell_line_name","synonyms"]].copy()
prim = cello[["cellosaurus_accession","cellosaurus_cell_line_name"]].rename(
        columns={"cellosaurus_cell_line_name":"nm"})
syn  = (cello.assign(nm=cello["synonyms"].astype("string").str.split(r"[;|]"))
             .explode("nm")[["cellosaurus_accession","nm"]])
lookup = pd.concat([prim, syn], ignore_index=True).dropna(subset=["nm"])
lookup["key"] = _norm(lookup["nm"])
lookup = lookup.dropna(subset=["key"]).drop_duplicates("key")

# attempt match on the 108 raw names
rec = geo_info[geo_info["model_id"].isna() & geo_info["cellosaurus_id"].isna()].copy()
rec["key"] = _norm(rec["cell_line"])
hits = rec.merge(lookup, on="key", how="left")

matched = hits[hits["cellosaurus_accession"].notna()]
print(f"resolved to a cvcl via name/synonym: {matched['cell_line'].nunique()} unique names")
print(matched[["cell_line","cellosaurus_accession","nm"]].drop_duplicates().to_string())

resolved to a cvcl via name/synonym: 6 unique names
   cell_line cellosaurus_accession                                 nm
8       c106             cvcl_ei67              c106 [human melanoma]
9       cc20             cvcl_2h54  cc20 [human colon adenocarcinoma]
23       d24             cvcl_w871               d24 [human melanoma]
24       d22             cvcl_3458                          d22 [dog]
25       d38             cvcl_hf99            d38 [human leukoplakia]
90        bj             cvcl_e483              bj [human b-cell ihw]


In [115]:
si_cvcl = set(sample_info[rrid_col].dropna()
              .str.replace(r"(?i)^rrid:","",regex=True).str.strip().str.lower())
matched_in_si = matched[matched["cellosaurus_accession"].str.lower().isin(si_cvcl)]
print(f"of the 6, actually in sample_info: {matched_in_si['cell_line'].nunique()}")
print(matched_in_si[["cell_line","cellosaurus_accession","nm"]].drop_duplicates().to_string())

of the 6, actually in sample_info: 0
Empty DataFrame
Columns: [cell_line, cellosaurus_accession, nm]
Index: []


In [116]:
import numpy as np
geo_info["link_status"] = np.select(
    [geo_info["model_id"].notna(),
     geo_info["model_id"].isna() & geo_info["cellosaurus_id"].notna()],
    ["modelid_linked", "expression_only"],
    default="unmatched")
print(geo_info["link_status"].value_counts())

geo_info.to_parquet(out / "geo_info.parquet", engine="fastparquet", index=False)
print(f"saved geo_info: {geo_info.shape}")

link_status
modelid_linked     2308
expression_only     851
unmatched           108
Name: count, dtype: int64
saved geo_info: (3267, 25)


In [117]:
geo_info

,geo_accession,cel_file_names,title,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,...,gse_filename,cell_line,disease,origin,cellosaurus_id,cellline,matching_type,model_id,omics_linked,link_status
0,gsm101610,gsm101610,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_0131,a172,cello geo gsm,ach-000558,True,modelid_linked
1,gsm101615,gsm101615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm,ach-000595,True,modelid_linked
2,gsm101616,gsm101616,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_0393,ln229,cello geo gsm,ach-000595,True,modelid_linked
3,gsm101667,gsm101667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,ach-000437,True,modelid_linked
4,gsm101668,gsm101668,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_1715,sw1088,cello geo gsm,ach-000437,True,modelid_linked
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3262,gsm960294,gsm960294,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN,False,expression_only
3263,gsm960295,gsm960295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN,False,expression_only
3264,gsm960296,gsm960296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_0038,hacat,cello geo gsm,NaN,False,expression_only
3265,gsm960297,gsm960297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,cvcl_7082,<NA>,NaN,NaN,False,expression_only
